# Figure 2 — ASReview screening trajectory from public derived ledger

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p/'requirements.txt').exists() and (p/'data').is_dir(): return p
    raise RuntimeError('Repository root not found. Run from inside the cloned repository.')
ROOT=find_repo_root()
print('Repository root:', ROOT)

In [ ]:
import matplotlib.pyplot as plt
ledger=pd.read_csv(ROOT/'data/Figure2/Figure2_ScreeningLedger_DERIVED_765.csv')
seeds=pd.read_csv(ROOT/'data/Figure2/Figure2_PositiveSeeds_DERIVED_2.csv')
config=pd.read_csv(ROOT/'data/Figure2/Figure2_ASReview_config.csv')
assert len(ledger)==765 and ledger.Screening_position.tolist()==list(range(1,766))
assert len(seeds)==2 and seeds.Role.eq('Positive seed').all()
assert int(ledger.Label_numeric.sum())==50
assert int((ledger.Label_numeric==0).sum())==715
last_rel=int(ledger.loc[ledger.Label_numeric.eq(1),'Screening_position'].max())
assert last_rel==408 and 765-last_rel==357
assert 765+2==767 and 50+2==52
ledger['Cumulative_relevant']=ledger.Label_numeric.cumsum()
blocks=ledger.assign(block=((ledger.Screening_position-1)//50)+1).groupby('block').agg(screened=('Broad_ID','size'),relevant=('Label_numeric','sum')).reset_index()
fig,ax=plt.subplots(figsize=(6,4)); ax.plot(ledger.Screening_position,ledger.Cumulative_relevant); ax.set(xlabel='Human-screened records',ylabel='Cumulative relevant'); plt.close(fig)
print('Figure 2 ASReview derived-ledger validation: PASS')
print({'human_screened':765,'human_relevant':50,'human_not_relevant':715,'positive_seeds':2,'total_labeled':767,'full_text_candidates':52,'last_relevant_position':408,'terminal_not_relevant':357})